# WatchNext Models

This notebook integrates the model modules from `watchnext/models` into one reproducible workflow. It loads the dataset, trains the available recommenders, inspects saved artifacts, and demonstrates sample recommendations from each model family.

## Coverage
1. Load the MovieLens data through the project loader.
2. Run the popularity, item-based, content-based, collaborative filtering, neural CF, and hybrid recommenders.
3. Inspect the files written to `models/saved`.
4. Optionally run the evaluation pipeline.

## 1. Setup and Imports

The notebook imports the project modules directly so the Python package remains the single source of truth.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 120)

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "watchnext").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from watchnext.common import get_paths
from watchnext.data_loader import load_raw_data
from watchnext.evaluation.evaluate_models import evaluate_models
from watchnext.models.collaborative_filtering import (
    SURPRISE_AVAILABLE,
    score_user_items as cf_score_user_items,
    train_collaborative_models,
)
from watchnext.models.content_based import recommend_similar, train_content_model
from watchnext.models.hybrid_recommender import HybridRecommender
from watchnext.models.neural_cf import score_user_items as neural_score_user_items
from watchnext.models.neural_cf import train_neural_cf
from watchnext.models.popularity_model import get_popular_movies

In [2]:
paths = get_paths()
paths_frame = pd.DataFrame(
    {
        "path": [
            paths.raw_data,
            paths.plots,
            paths.processed,
            paths.models,
            paths.saved_models,
            paths.reports,
        ]
    },
    index=["raw_data", "plots", "processed", "models", "saved_models", "reports"],
)
display(paths_frame)
print(f"Collaborative filtering backend: {'surprise' if SURPRISE_AVAILABLE else 'sklearn fallback'}")

,path
raw_data,D:\Projects\Data_Science\WatchNext\data\ml-latest-small
plots,D:\Projects\Data_Science\WatchNext\data\plots
processed,D:\Projects\Data_Science\WatchNext\data\processed
models,D:\Projects\Data_Science\WatchNext\models
saved_models,D:\Projects\Data_Science\WatchNext\models\saved
reports,D:\Projects\Data_Science\WatchNext\reports


Collaborative filtering backend: sklearn fallback


## 2. Load and Inspect the Dataset

The loader will reuse local CSV files when present and can download the MovieLens Small dataset automatically when they are missing.

In [4]:
dataset = load_raw_data()
movies = dataset["movies"]
ratings = dataset["ratings"]
tags = dataset["tags"]
links = dataset["links"]

summary = pd.DataFrame(
    {
        "rows": {name: frame.shape[0] for name, frame in dataset.items()},
        "columns": {name: frame.shape[1] for name, frame in dataset.items()},
    }
)
display(summary)

print(f"Unique users: {ratings['userId'].nunique()}")
print(f"Unique movies: {ratings['movieId'].nunique()}")

,rows,columns
movies,9742,3
ratings,100836,4
links,9742,3
tags,3683,4


Unique users: 610
Unique movies: 9724


In [5]:
display(movies.head())
display(ratings.head())
display(tags.head())

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


## 3. Popularity-Based Recommendations

This section calls the helper defined in `popularity_model.py` to retrieve high-rated movies with a minimum number of ratings.

In [6]:
popular_movies = get_popular_movies(movies, ratings, min_ratings=50)
display(popular_movies)

,title,count,mean
0,"Shawshank Redemption, The (1994)",317,4.429022
1,"Godfather, The (1972)",192,4.289062
2,Fight Club (1999),218,4.272936
3,Cool Hand Luke (1967),57,4.271930
4,Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964),97,4.268041
5,Rear Window (1954),84,4.261905
6,"Godfather: Part II, The (1974)",129,4.259690
7,"Departed, The (2006)",107,4.252336
8,Goodfellas (1990),126,4.250000
9,Casablanca (1942),100,4.240000


## 4. Content-Based Recommender

Training the content model writes similarity and metadata artifacts into `models/saved`.

In [7]:
content_artifacts = train_content_model()
display(pd.Series(content_artifacts, name="artifact_path").to_frame())

,artifact_path
similarity,D:\Projects\Data_Science\WatchNext\models\saved\content_similarity.pkl
movies,D:\Projects\Data_Science\WatchNext\models\saved\content_movies.pkl
vectorizer,D:\Projects\Data_Science\WatchNext\models\saved\content_vectorizer.pkl


In [8]:
seed_movie_id = int(movies.iloc[0]["movieId"])
seed_title = movies.loc[movies["movieId"] == seed_movie_id, "title"].iloc[0]
print(f"Seed movie: {seed_title} ({seed_movie_id})")
display(pd.DataFrame(recommend_similar(seed_movie_id, top_n=10)))

Seed movie: Toy Story (1995) (1)


,movieId,title,score
0,3114,Toy Story 2 (1999),0.929544
1,2294,Antz (1998),0.912871
2,3754,"Adventures of Rocky and Bullwinkle, The (2000)",0.912871
3,4886,"Monsters, Inc. (2001)",0.912871
4,45074,"Wild, The (2006)",0.912871
5,53121,Shrek the Third (2007),0.912871
6,65577,"Tale of Despereaux, The (2008)",0.912871
7,91355,Asterix and the Vikings (Astérix et les Vikings) (2006),0.912871
8,103755,Turbo (2013),0.912871
9,136016,The Good Dinosaur (2015),0.912871


## 5. Collaborative Filtering

The training function chooses its backend automatically. In the current environment, NumPy 2.x means the notebook will usually use the sklearn fallback rather than `surprise`.

In [9]:
cf_artifacts = train_collaborative_models(tune=False)
display(pd.Series(cf_artifacts, name="artifact_path").to_frame())

,artifact_path
svd,D:\Projects\Data_Science\WatchNext\models\saved\cf_svd.pkl
nmf,D:\Projects\Data_Science\WatchNext\models\saved\cf_nmf.pkl
metadata,D:\Projects\Data_Science\WatchNext\models\saved\cf_metadata.json


In [10]:
sample_user_id = int(ratings["userId"].value_counts().index[0])
seen_movie_ids = set(ratings.loc[ratings["userId"] == sample_user_id, "movieId"])
candidate_ids = [movie_id for movie_id in movies["movieId"].tolist() if movie_id not in seen_movie_ids]

print(f"Sample user: {sample_user_id}")
print(f"Seen movies: {len(seen_movie_ids)}")
print(f"Candidate movies: {len(candidate_ids)}")

Sample user: 414
Seen movies: 2698
Candidate movies: 7044


In [11]:
cf_scores = cf_score_user_items(sample_user_id, candidate_ids, algorithm="svd")
cf_top = cf_scores.sort_values(ascending=False).head(10)
cf_recommendations = movies[movies["movieId"].isin(cf_top.index)].copy()
cf_recommendations["score"] = cf_recommendations["movieId"].map(cf_top.to_dict())
display(cf_recommendations.sort_values("score", ascending=False)[["movieId", "title", "score"]])

,movieId,title,score
9128,146656,Creed (2015),0.358708
8179,102903,Now You See Me (2013),0.358666
444,509,"Piano, The (1993)",0.344650
1113,1449,Waiting for Guffman (1996),0.327249
628,800,Lone Star (1996),0.321524
889,1186,"Sex, Lies, and Videotape (1989)",0.302495
4046,5747,Gallipoli (1981),0.293958
8696,122920,Captain America: Civil War (2016),0.293051
3129,4211,Reversal of Fortune (1990),0.288408
4397,6461,"Unforgiven, The (1960)",0.282204


## 6. Neural Collaborative Filtering

This model learns user and item embeddings in PyTorch. The parameter cell makes it easy to shorten training while iterating in the notebook.

In [12]:
NEURAL_EPOCHS = 3
NEURAL_BATCH_SIZE = 1024

In [13]:
neural_artifacts = train_neural_cf(epochs=NEURAL_EPOCHS, batch_size=NEURAL_BATCH_SIZE)
display(
    pd.DataFrame(
        {
            "artifact_path": [neural_artifacts.model_path, neural_artifacts.metadata_path]
        },
        index=["model", "metadata"],
    )
)

,artifact_path
model,D:\Projects\Data_Science\WatchNext\models\saved\neural_cf.pt
metadata,D:\Projects\Data_Science\WatchNext\models\saved\neural_cf_meta.pkl


In [14]:
neural_scores = neural_score_user_items(sample_user_id, candidate_ids)
neural_top = neural_scores.sort_values(ascending=False).head(10)
neural_recommendations = movies[movies["movieId"].isin(neural_top.index)].copy()
neural_recommendations["score"] = neural_recommendations["movieId"].map(neural_top.to_dict())
display(neural_recommendations.sort_values("score", ascending=False)[["movieId", "title", "score"]])

,movieId,title,score
2394,3176,"Talented Mr. Ripley, The (1999)",0.612952
209,243,Gordy (1995),0.606277
3009,4025,Miss Congeniality (2000),0.603297
5735,30793,Charlie and the Chocolate Factory (2005),0.599282
5994,37380,Doom (2005),0.599036
1786,2384,Babe: Pig in the City (1998),0.597683
25,26,Othello (1995),0.591156
6163,44399,"Shaggy Dog, The (2006)",0.589718
6711,58627,Never Back Down (2008),0.589609
1936,2567,EDtv (1999),0.587970


## 7. Hybrid Recommender

The hybrid recommender combines collaborative, content, and neural scores after the required artifacts are available.

In [15]:
hybrid = HybridRecommender()
hybrid_recommendations = pd.DataFrame(hybrid.recommend(sample_user_id, top_n=10))
display(hybrid_recommendations)

config_path = hybrid.save_config()
print(f"Hybrid config saved to: {config_path}")

,movieId,title,score
0,2352,"Big Chill, The (1983)",0.910602
1,2926,Hairspray (1988),0.851282
2,1186,"Sex, Lies, and Videotape (1989)",0.839059
3,146656,Creed (2015),0.837710
4,2583,Cookie's Fortune (1999),0.823816
5,6001,"King of Comedy, The (1983)",0.819840
6,281,Nobody's Fool (1994),0.819005
7,5890,Elling (2001),0.816311
8,2596,SLC Punk! (1998),0.815924
9,3246,Malcolm X (1992),0.809209


Hybrid config saved to: D:\Projects\Data_Science\WatchNext\models\saved\hybrid_config.json


## 8. Saved Artifacts

This quick check helps confirm that the training functions wrote the expected files.

In [16]:
saved_files = sorted(paths.saved_models.glob("*"))
display(pd.DataFrame({"file": [file.name for file in saved_files], "path": saved_files}))

,file,path
0,cf_metadata.json,D:\Projects\Data_Science\WatchNext\models\saved\cf_metadata.json
1,cf_nmf.pkl,D:\Projects\Data_Science\WatchNext\models\saved\cf_nmf.pkl
2,cf_svd.pkl,D:\Projects\Data_Science\WatchNext\models\saved\cf_svd.pkl
3,content_movies.pkl,D:\Projects\Data_Science\WatchNext\models\saved\content_movies.pkl
4,content_similarity.pkl,D:\Projects\Data_Science\WatchNext\models\saved\content_similarity.pkl
5,content_vectorizer.pkl,D:\Projects\Data_Science\WatchNext\models\saved\content_vectorizer.pkl
6,hybrid_config.json,D:\Projects\Data_Science\WatchNext\models\saved\hybrid_config.json
7,neural_cf.pt,D:\Projects\Data_Science\WatchNext\models\saved\neural_cf.pt
8,neural_cf_meta.pkl,D:\Projects\Data_Science\WatchNext\models\saved\neural_cf_meta.pkl


## 9. Item-Based Collaborative Filtering

The package file for item-based collaborative filtering is still empty, so this notebook builds the model directly from the ratings matrix. It computes item-item cosine similarity, saves the similarity artifact, and stores the user-item matrix needed for later scoring.

In [17]:
import json
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [18]:
def train_item_based_model(
    ratings: pd.DataFrame,
    output_dir: Path,
    min_item_ratings: int = 5,
) -> dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)

    rating_counts = ratings.groupby("movieId").size()
    eligible_movie_ids = rating_counts[rating_counts >= min_item_ratings].index.tolist()
    filtered_ratings = ratings[ratings["movieId"].isin(eligible_movie_ids)].copy()

    user_item_matrix = filtered_ratings.pivot_table(
        index="userId",
        columns="movieId",
        values="rating",
        fill_value=0.0,
    )

    item_user_matrix = user_item_matrix.T
    similarity = cosine_similarity(item_user_matrix).astype(np.float32)
    similarity_df = pd.DataFrame(
        similarity,
        index=item_user_matrix.index,
        columns=item_user_matrix.index,
    )

    similarity_path = output_dir / "item_based_similarity.pkl"
    matrix_path = output_dir / "item_based_user_item_matrix.pkl"
    metadata_path = output_dir / "item_based_metadata.json"

    similarity_df.to_pickle(similarity_path)
    user_item_matrix.to_pickle(matrix_path)

    metadata = {
        "min_item_ratings": int(min_item_ratings),
        "num_users": int(user_item_matrix.shape[0]),
        "num_items": int(user_item_matrix.shape[1]),
        "num_ratings": int(len(filtered_ratings)),
    }
    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2)

    return {
        "similarity": similarity_path,
        "user_item_matrix": matrix_path,
        "metadata": metadata_path,
    }


def recommend_similar_items_from_artifact(
    movie_id: int,
    movies: pd.DataFrame,
    model_dir: Path,
    top_n: int = 10,
) -> pd.DataFrame:
    similarity_df = pd.read_pickle(model_dir / "item_based_similarity.pkl")
    if movie_id not in similarity_df.index:
        return pd.DataFrame(columns=["movieId", "title", "score"])

    ranked = similarity_df.loc[movie_id].sort_values(ascending=False).iloc[1 : top_n + 1]
    recommendations = movies[movies["movieId"].isin(ranked.index)].copy()
    recommendations["score"] = recommendations["movieId"].map(ranked.to_dict())
    return recommendations.sort_values("score", ascending=False)[["movieId", "title", "score"]]

In [19]:
ITEM_BASED_MIN_ITEM_RATINGS = 5

item_based_artifacts = train_item_based_model(
    ratings=ratings,
    output_dir=paths.saved_models,
    min_item_ratings=ITEM_BASED_MIN_ITEM_RATINGS,
)
display(pd.Series(item_based_artifacts, name="artifact_path").to_frame())

,artifact_path
similarity,D:\Projects\Data_Science\WatchNext\models\saved\item_based_similarity.pkl
user_item_matrix,D:\Projects\Data_Science\WatchNext\models\saved\item_based_user_item_matrix.pkl
metadata,D:\Projects\Data_Science\WatchNext\models\saved\item_based_metadata.json


In [20]:
sample_item_id = int(movies.iloc[0]["movieId"])
sample_item_title = movies.loc[movies["movieId"] == sample_item_id, "title"].iloc[0]
print(f"Sample item: {sample_item_title} ({sample_item_id})")
display(recommend_similar_items_from_artifact(sample_item_id, movies, paths.saved_models, top_n=10))

Sample item: Toy Story (1995) (1)


,movieId,title,score
2355,3114,Toy Story 2 (1999),0.572601
418,480,Jurassic Park (1993),0.565637
615,780,Independence Day (a.k.a. ID4) (1996),0.564262
224,260,Star Wars: Episode IV - A New Hope (1977),0.557388
314,356,Forrest Gump (1994),0.547096
322,364,"Lion King, The (1994)",0.541145
911,1210,Star Wars: Episode VI - Return of the Jedi (1983),0.541089
546,648,Mission: Impossible (1996),0.538913
964,1265,Groundhog Day (1993),0.534169
969,1270,Back to the Future (1985),0.530381


## 10. Simple Evaluation

Run the package evaluation pipeline to compare model behavior on a temporal split. This can take a little longer than the earlier demonstration cells.

In [21]:
evaluation_results = evaluate_models()
display(pd.DataFrame(evaluation_results).T)

,precision_at_k,recall_at_k,ndcg_at_k,coverage,novelty,rmse,mae
content,0.034,0.004680,0.043354,0.021146,13.947202,NaN,NaN
cf,0.545,0.123431,0.566815,0.020427,9.484505,1.757956,1.25156
hybrid,0.115,0.052115,0.122426,0.021248,14.850448,NaN,NaN
